In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from datetime import datetime, timedelta
from sqlalchemy import select
from decimal import Decimal
import time
from loguru import logger

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_validate, cross_val_score
from sklearn.base import clone
from probatus.feature_elimination import ShapRFECV, EarlyStoppingShapRFECV
from xgboost import XGBClassifier
import shap


base_dir = Path("__file__").resolve().parents[2]
sys.path.insert(0, str(base_dir))

/home/zakihran/.cache/pypoetry/virtualenvs/idx-portfolio-pipeline-ne9i8t3d-py3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from data.db.client import init_async_db
from data.db import entities_transformed as et
from data.db.statements import fetch_frame
from stock_market.config import SECTOR_MAPPING, HORIZON

# Analytics

## **training**

### **data preprocessing**

In [3]:
client = init_async_db()
stock_ids = []

start_date = datetime(2020,1,1)
end_date = datetime(2023,12,31)

In [4]:
# Fetch history for rolling features and future bars for the label horizon.
fetch_start_date = start_date - timedelta(days=400)
fetch_end_date = end_date + timedelta(days=60)

async with client.connect() as app:
    stmt = select(
        et.PriceData.stock_id,
        et.PriceData.trade_date,
        et.PriceData.high,
        et.PriceData.low,
        et.PriceData.close,
    ).where(
        et.PriceData.trade_date.between(fetch_start_date, fetch_end_date)
    )

    data = await app.execute(stmt)
    data = pd.DataFrame(data.mappings().all())


    metadata_stmt = select(
        et.StockMetadata.stock_id,
        et.StockMetadata.sector
    ).where(
        et.StockMetadata.sector.is_not(None)
    )

    metadata = await app.execute(metadata_stmt)
    metadata = pd.DataFrame(metadata.mappings().all())
    metadata["sector"] = (metadata["sector"].map(SECTOR_MAPPING))

    data = pd.merge(
        data,
        metadata,
        how="inner",
        on="stock_id"
    )

In [11]:
def make_triple_barrier_labels(
    data: pd.DataFrame,
    horizon: int = HORIZON,
    upper_return: float = 0.05,
    lower_return: float = 0.05,
) -> pd.Series:
    """Create 0/1/2 lower/neither/upper labels for rows sorted by stock and date.
    A bar touching both barriers is labeled lower; incomplete windows remain NA.
    """
    required = {"stock_id", "close", "high", "low"}
    missing = required.difference(data.columns)
    if missing:
        raise ValueError(f"Missing columns for triple-barrier labels: {sorted(missing)}")
    if horizon < 1:
        raise ValueError("horizon must be at least 1")
    if (
        not np.isfinite(upper_return)
        or not np.isfinite(lower_return)
        or upper_return <= 0
        or not 0 < lower_return < 1
    ):
        raise ValueError("upper_return must be positive and lower_return must be between 0 and 1")

    close = pd.to_numeric(data["close"], errors="coerce").to_numpy(dtype="float64")
    high = pd.to_numeric(data["high"], errors="coerce").to_numpy(dtype="float64")
    low = pd.to_numeric(data["low"], errors="coerce").to_numpy(dtype="float64")
    labels = np.full(len(data), np.nan, dtype="float64")

    for positions in data.groupby("stock_id", sort=False).indices.values():
        positions = np.asarray(positions)
        event_count = len(positions) - horizon
        if event_count <= 0:
            continue

        event_positions = positions[:event_count]
        entry_prices = close[event_positions]
        active = np.isfinite(entry_prices) & (entry_prices > 0)
        upper_barriers = entry_prices * (1 + upper_return)
        lower_barriers = entry_prices * (1 - lower_return)

        for offset in range(1, horizon + 1):
            future_positions = positions[offset : offset + event_count]
            valid_bar = np.isfinite(high[future_positions]) & np.isfinite(low[future_positions])
            active[~valid_bar] = False

            upper_hit = active & (high[future_positions] >= upper_barriers)
            lower_hit = active & (low[future_positions] <= lower_barriers)
            touched = upper_hit | lower_hit
            labels[event_positions[touched]] = np.where(lower_hit[touched], 0, 2)
            active[touched] = False

        labels[event_positions[active]] = 1

    return pd.Series(pd.array(labels, dtype="Int8"), index=data.index, name="label")


def process_data_training(
    data: pd.DataFrame,
    start_date: datetime | None = None,
    end_date: datetime | None = None,
    horizon: int = HORIZON,
    upper_return: float = 0.05,
    lower_return: float = 0.05,
    non_overlapping: bool = True,
) -> pd.DataFrame:
    required = {"stock_id", "trade_date", "close", "high", "low"}
    missing = required.difference(data.columns)
    if missing:
        raise ValueError(f"Missing columns for training data: {sorted(missing)}")

    data = data.sort_values(["stock_id", "trade_date"]).reset_index(drop=True).copy()
    for column in ("close", "high", "low"):
        data[column] = pd.to_numeric(data[column], errors="coerce")

    by_stock = data.groupby("stock_id", sort=False)
    close = by_stock["close"]
    data["returns"] = close.pct_change(fill_method=None)
    data["moving_avg_30d"] = close.rolling(30, min_periods=30).mean().reset_index(level=0, drop=True)
    data["distance_moving_avg"] = data["close"].div(data["moving_avg_30d"]).sub(1).astype("float32")
    data["high_52w"] = by_stock["high"].transform(
        lambda values: values.shift(1).rolling(252, min_periods=252).max()
    )
    data["low_52w"] = by_stock["low"].transform(
        lambda values: values.shift(1).rolling(252, min_periods=252).min()
    )

    for period in (5, 10, 20, 30, 120, 250):
        data[f"momentum_{period}d"] = close.pct_change(period, fill_method=None).astype("float32")

    for period in range(5, 31, 5):
        daily_volatility = (
            data.groupby("stock_id", sort=False)["returns"]
            .rolling(period, min_periods=period)
            .std()
            .reset_index(level=0, drop=True)
        )
        data[f"volatility_{period}d_daily"] = daily_volatility
        data[f"volatility_{period}d_ann"] = daily_volatility * np.sqrt(252)

    for period in (3, 5, 10, 15):
        prior_average = data.groupby("stock_id", sort=False)["moving_avg_30d"].shift(period)
        data[f"moving_avg_slope_{period}d"] = data["moving_avg_30d"].div(prior_average).sub(1)

    data["distance_high_52w"] = data["close"].div(data["high_52w"]).sub(1).astype("float32")
    data["distance_low_52w"] = data["close"].div(data["low_52w"]).sub(1).astype("float32")
    data["risk_adjusted_momentum"] = data["momentum_30d"].div(
        data["volatility_30d_ann"].replace(0, np.nan)
    )

    data["label"] = make_triple_barrier_labels(
        data,
        horizon=horizon,
        upper_return=upper_return,
        lower_return=lower_return,
    )

    if start_date is not None:
        data = data.loc[pd.to_datetime(data["trade_date"]) >= pd.Timestamp(start_date)]
    if end_date is not None:
        data = data.loc[pd.to_datetime(data["trade_date"]) <= pd.Timestamp(end_date)]

    if non_overlapping:
        # One decision per stock every `horizon` trading days, so label windows never overlap.
        data = data.loc[data.groupby("stock_id", sort=False).cumcount() % horizon == 0]

    return data.dropna().assign(label=lambda frame: frame["label"].astype("int8")).copy()

def purged_time_split(df, train_frac=0.6, val_frac=0.2, date_col: str = "trade_date"):
    """Chronological split by date. Rows whose label window reaches into the next
    split are dropped, so no train/val label overlaps val/test dates."""
    d = df.dropna(subset=["label"]).copy()
    udates = np.sort(d[date_col].unique())
    val_start = udates[int(len(udates) * train_frac)]
    test_start = udates[int(len(udates) * (train_frac + val_frac))]

    train = d[(d[date_col] < val_start) & (d["label_end"] < val_start)]
    val = d[(d[date_col] >= val_start) & (d[date_col] < test_start) & (d["label_end"] < test_start)]
    test = d[d[date_col] >= test_start]
    return train, val, test

In [6]:
indexes = process_data_training(data, start_date=start_date, end_date=end_date)

indexes = indexes[[
    "stock_id", "trade_date", "sector", "returns", "distance_moving_avg",
    'momentum_5d', 'momentum_10d', 'momentum_20d', 'momentum_30d',
    'momentum_120d', 'momentum_250d', 'volatility_5d_ann', 'volatility_10d_ann',
    'volatility_15d_ann', 'volatility_20d_ann', 'volatility_25d_ann', 'volatility_30d_ann',
    'moving_avg_slope_3d', 'moving_avg_slope_5d', 'moving_avg_slope_10d', 'moving_avg_slope_15d',
    'distance_high_52w', 'distance_low_52w', 'risk_adjusted_momentum', 'label'
]].copy()


dec_cols = [
    c for c in indexes.select_dtypes(include="object").columns
    if indexes[c].map(lambda v: isinstance(v, Decimal)).any()
]
indexes[dec_cols] = indexes[dec_cols].astype("float64")
indexes["sector"] = indexes["sector"].astype("category")  # if it holds floats like 3.0

In [7]:
print(indexes["label"].value_counts(normalize=True))

label
2    0.347987
0    0.346593
1    0.305420
Name: proportion, dtype: float64


In [7]:
features_list = [
    "sector", "returns", "distance_moving_avg",
    'momentum_5d', 'momentum_10d', 'momentum_20d', 'momentum_30d',
    'momentum_120d', 'momentum_250d', 'volatility_5d_ann', 'volatility_10d_ann',
    'volatility_15d_ann', 'volatility_20d_ann', 'volatility_25d_ann', 'volatility_30d_ann',
    'moving_avg_slope_3d', 'moving_avg_slope_5d', 'moving_avg_slope_10d', 'moving_avg_slope_15d',
    'distance_high_52w', 'distance_low_52w', 'risk_adjusted_momentum'
]
meta = indexes[["stock_id", "trade_date"]]  # kept aside for splitting and analysis

X = indexes[features_list]
y = indexes["label"]


cutoff = indexes["trade_date"].quantile(0.8)           # or a specific date
gap = pd.Timedelta(days=HORIZON)                       # your barrier horizon

train_mask = indexes["trade_date"] < cutoff - gap      # purge/embargo
val_mask   = indexes["trade_date"] >= cutoff

X_train, y_train = X[train_mask], y[train_mask]
X_val,   y_val   = X[val_mask],   y[val_mask]

/tmp/ipykernel_511589/1729846488.py:16: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  gap = pd.Timedelta(days=HORIZON)                       # your barrier horizon


### **feature selection**

In [27]:
# xgb = XGBClassifier(
#     n_estimators=300,
#     learning_rate=0.1, 
#     max_depth=6,
#     objective="multi:softprob",
#     num_class=3,
#     tree_method="hist",
#     device="cuda",
#     random_state=42,
#     enable_categorical=True,
# )


# shap_eliminator = ShapRFECV(
#     model=xgb,
#     step=0.1, 
#     min_features_to_select=10,
#     cv=5,  
#     scoring="roc_auc_ovr",
#     n_jobs=1,
#     verbose=2
# )



# report = shap_eliminator.fit_compute(
#     X_train,
#     y_train
# )

In [ ]:
def shap_rfe(model, X, y, step=0.1, min_features=10, cv=3):
    feats = list(X.columns)
    rows = []

    skf = StratifiedKFold(
        n_splits=cv,
        shuffle=True,
        random_state=42,
    )

    iteration = 1
    total_start = time.perf_counter()

    logger.info(
        "Starting SHAP-RFE | Initial Features={} | Min Features={} | Step={} | CV={}",
        len(feats),
        min_features,
        step,
        cv,
    )

    while True:
        iter_start = time.perf_counter()

        logger.info(
            "[Iteration {}] Evaluating {} features...",
            iteration,
            len(feats),
        )

        score = cross_val_score(
            clone(model),
            X[feats],
            y,
            cv=skf,
            scoring="f1_macro",
            n_jobs=-1,
        ).mean()

        rows.append((len(feats), score, list(feats)))

        logger.info(
            "[Iteration {}] CV F1 Macro = {:.6f}",
            iteration,
            score,
        )

        if len(feats) <= min_features:
            logger.success(
                "Reached minimum feature count ({}). Stopping.",
                min_features,
            )
            break

        logger.info(
            "[Iteration {}] Training model for SHAP importance...",
            iteration,
        )

        m = clone(model).fit(X[feats], y)

        logger.info(
            "[Iteration {}] Computing SHAP values...",
            iteration,
        )

        sv = shap.TreeExplainer(m).shap_values(X[feats])

        if isinstance(sv, list):
            sv = np.stack(sv, axis=-1)

        imp = np.abs(sv).mean(axis=(0, 2))

        drop = max(1, int(len(feats) * step))
        keep_n = max(min_features, len(feats) - drop)

        ranked = np.argsort(imp)[::-1]
        removed = [feats[i] for i in ranked[keep_n:]]

        logger.info(
            "[Iteration {}] Removing {} feature(s): {}",
            iteration,
            len(removed),
            ", ".join(removed[:10]) + ("..." if len(removed) > 10 else ""),
        )

        feats = [feats[i] for i in ranked[:keep_n]]

        logger.info(
            "[Iteration {}] Remaining features: {} | Elapsed: {:.2f}s",
            iteration,
            len(feats),
            time.perf_counter() - iter_start,
        )

        iteration += 1

    logger.success(
        "SHAP-RFE completed in {:.2f}s after {} iteration(s). Final feature count: {}",
        time.perf_counter() - total_start,
        iteration,
        len(feats),
    )

    return rows


xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.1, 
    max_depth=5,
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    device="cuda",
    random_state=42,
    enable_categorical=True,
)


rows = shap_rfe(xgb, X_train, y_train, min_features=8)
best = max(rows, key=lambda r: r[1])
selected_features = best[2]

2026-11-05 21:33:37.453 | INFO     | __main__:shap_rfe:14 - Starting SHAP-RFE | Initial Features=22 | Min Features=8 | Step=0.1 | CV=3
2026-11-05 21:33:37.454 | INFO     | __main__:shap_rfe:25 - [Iteration 1] Evaluating 22 features...
/home/zakihran/.cache/pypoetry/virtualenvs/idx-portfolio-pipeline-ne9i8t3d-py3.13/lib/python3.13/site-packages/xgboost/core.py:569: UserWarning: [21:34:13] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
/home/zakihran/.cache/pypoetry/virtualenvs/idx-portfolio-pipeline-ne9i8t3d-py3.13/lib/python3.13/site-packages/xgboost/core.py:569: UserWar

In [9]:
best

(20,
 np.float64(0.5180453055858588),
 ['volatility_30d_ann',
  'distance_high_52w',
  'distance_low_52w',
  'returns',
  'volatility_10d_ann',
  'volatility_20d_ann',
  'momentum_120d',
  'volatility_5d_ann',
  'volatility_25d_ann',
  'momentum_5d',
  'volatility_15d_ann',
  'sector',
  'distance_moving_avg',
  'momentum_250d',
  'moving_avg_slope_15d',
  'moving_avg_slope_3d',
  'momentum_10d',
  'momentum_20d',
  'moving_avg_slope_10d',
  'moving_avg_slope_5d'])

### **model training**

In [10]:
start_date = datetime(2018,1,1)
end_date = datetime(2023,12,31)

indexes = process_data_training(data, start_date=start_date, end_date=end_date)

indexes = indexes[[
    "stock_id", "trade_date", "sector", "returns", "distance_moving_avg",
    'momentum_5d', 'momentum_10d', 'momentum_20d', 'momentum_30d',
    'momentum_120d', 'momentum_250d', 'volatility_5d_ann', 'volatility_10d_ann',
    'volatility_15d_ann', 'volatility_20d_ann', 'volatility_25d_ann', 'volatility_30d_ann',
    'moving_avg_slope_3d', 'moving_avg_slope_5d', 'moving_avg_slope_10d', 'moving_avg_slope_15d',
    'distance_high_52w', 'distance_low_52w', 'risk_adjusted_momentum', 'label'
]].copy()


dec_cols = [
    c for c in indexes.select_dtypes(include="object").columns
    if indexes[c].map(lambda v: isinstance(v, Decimal)).any()
]
indexes[dec_cols] = indexes[dec_cols].astype("float64")
indexes["sector"] = indexes["sector"].astype("category")  # if it holds floats like 3.0

In [16]:
X = indexes[selected_features]
y = indexes["label"]


cutoff = indexes["trade_date"].quantile(0.8)           # or a specific date
gap = pd.Timedelta(days=HORIZON)                       # your barrier horizon

train_mask = indexes["trade_date"] < cutoff - gap      # purge/embargo
val_mask   = indexes["trade_date"] >= cutoff

X_train, y_train = X[train_mask], y[train_mask]
X_val,   y_val   = X[val_mask],   y[val_mask]

/tmp/ipykernel_511589/2213674251.py:6: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  gap = pd.Timedelta(days=HORIZON)                       # your barrier horizon


In [21]:
len(X_train)//3

11267

In [17]:
xgb = XGBClassifier(
    n_estimators=3000,
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    device="cuda",
    eval_metric="mlogloss",
    random_state=42,
    enable_categorical=True,
)


param_dist = {
    "max_depth": [3, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 10],
    "gamma": [0, 0.1, 0.5, 1],
    "reg_lambda": [1, 5, 10],
    "reg_alpha": [0, 0.1, 1],
}


search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=50,
    scoring="f1_macro",
    cv=3,
    random_state=42,
    verbose=2
)


search.fit(
    X_train,
    y_train,
)


print(search.best_params_)
print(search.best_score_)

Fitting 3 folds for each of 50 candidates, totalling 150 fits


/home/zakihran/.cache/pypoetry/virtualenvs/idx-portfolio-pipeline-ne9i8t3d-py3.13/lib/python3.13/site-packages/xgboost/core.py:569: UserWarning: [21:58:19] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[CV] END colsample_bytree=0.7, gamma=1, learning_rate=0.03, max_depth=3, min_child_weight=10, reg_alpha=0, reg_lambda=1, subsample=0.6; total time=  21.1s
[CV] END colsample_bytree=0.7, gamma=1, learning_rate=0.03, max_depth=3, min_child_weight=10, reg_alpha=0, reg_lambda=1, subsample=0.6; total time=  21.0s
[CV] END colsample_bytree=0.7, gamma=1, learning_rate=0.03, max_depth=3, min_child_weight=10, reg_alpha=0, reg_lambda=1, subsample=0.6; total time=  21.0s
[CV] END colsample_bytree=0.6, gamma=0, learning_rate=0.03, max_depth=6, min_child_weight=10, reg_alpha=0, reg_lambda=5, subsample=0.6; total time=  57.6s
[CV] END colsample_bytree=0.6, gamma=0, learning_rate=0.03, max_depth=6, min_child_weight=10, reg_alpha=0, reg_lambda=5, subsample=0.6; total time=  59.9s
[CV] END colsample_bytree=0.6, gamma=0, learning_rate=0.03, max_depth=6, min_child_weight=10, reg_alpha=0, reg_lambda=5, subsample=0.6; total time= 1.0min
[CV] END colsample_bytree=1.0, gamma=0.1, learning_rate=0.05, max_dept

In [18]:
best_params = search.best_params_
best_params

{'subsample': 0.8,
 'reg_lambda': 10,
 'reg_alpha': 1,
 'min_child_weight': 10,
 'max_depth': 3,
 'learning_rate': 0.01,
 'gamma': 0.5,
 'colsample_bytree': 0.8}

In [19]:
final_model = XGBClassifier(
    **best_params,
    random_state=42,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    tree_method="hist",
    device="cuda",
    enable_categorical=True,
    early_stopping_rounds=150,
)

final_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)


[0]	validation_0-mlogloss:1.10466
[50]	validation_0-mlogloss:1.04241
[99]	validation_0-mlogloss:1.01076


,"objective objective: typing.Union[str, xgboost.objective.Objective, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cuda'
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",150
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklear

In [20]:
artifact_dir = Path.joinpath(base_dir, "stock_market/analytics/models/scikit/")

final_model.save_model(artifact_dir / "outperformer_xgb_final3.ubj")

## **evaluate**

In [ ]:
# final_model = XGBClassifier()
# final_model.load_model("./models/scikit/outperformer_xgb_final.ubj")

In [21]:
# client = init_async_db()
# stock_ids = []

start_date = datetime(2024,1,1) + timedelta(days=400)
end_date = datetime(2026,1,31)

In [22]:
# Fetch history for rolling features and future bars for the label horizon.
fetch_start_date = start_date - timedelta(days=400)
fetch_end_date = end_date + timedelta(days=60)

async with client.connect() as app:
    stmt = select(
        et.PriceData.stock_id,
        et.PriceData.trade_date,
        et.PriceData.high,
        et.PriceData.low,
        et.PriceData.close,
    ).where(
        et.PriceData.trade_date.between(fetch_start_date, fetch_end_date)
    )

    data = await app.execute(stmt)
    data = pd.DataFrame(data.mappings().all())


    metadata_stmt = select(
        et.StockMetadata.stock_id,
        et.StockMetadata.sector
    )

    metadata = await app.execute(metadata_stmt)
    metadata = pd.DataFrame(metadata.mappings().all())
    metadata["sector"] = metadata["sector"].map(SECTOR_MAPPING)

    data = pd.merge(
        data,
        metadata,
        how="left",
        on="stock_id"
    )

In [23]:
indexes = process_data_training(data, start_date=start_date, end_date=end_date)

indexes = indexes[["label", *selected_features]].copy()


dec_cols = [
    c for c in indexes.select_dtypes(include="object").columns
    if indexes[c].map(lambda v: isinstance(v, Decimal)).any()
]
indexes[dec_cols] = indexes[dec_cols].astype("float64")

In [34]:
display(indexes.head())

,label,sector,returns,distance_moving_avg,momentum_5d,momentum_10d,momentum_20d,momentum_30d,momentum_120d,momentum_250d,...,volatility_20d_ann,volatility_25d_ann,volatility_30d_ann,moving_avg_slope_3d,moving_avg_slope_5d,moving_avg_slope_10d,moving_avg_slope_15d,distance_high_52w,distance_low_52w,risk_adjusted_momentum
257,1,3.0,-0.004274,-0.027411,-0.029167,-0.025105,-0.052845,-0.060484,0.030385,-0.153225,...,0.136456,0.224289,0.226016,-0.006360,-0.009919,-0.015749,-0.022310,-0.173759,0.123611,-0.267609
267,0,3.0,0.008621,-0.004961,0.044643,0.004292,-0.020920,-0.048780,-0.024811,-0.128254,...,0.163651,0.172438,0.163776,-0.006058,-0.008572,-0.018366,-0.028103,-0.170213,0.128433,-0.297848
277,2,3.0,-0.008968,-0.045494,-0.039130,-0.055556,-0.051502,-0.075314,-0.071345,-0.170740,...,0.309330,0.277163,0.256520,-0.007998,-0.010259,-0.015450,-0.023890,-0.216312,0.065743,-0.293598
287,0,3.0,-0.038462,-0.019038,-0.034335,0.018099,-0.038462,-0.034335,-0.120244,-0.133843,...,0.359378,0.337715,0.314545,-0.001741,-0.003909,-0.009358,-0.019521,-0.202128,0.085032,-0.109157
297,2,3.0,0.033175,-0.038094,0.004608,-0.031111,-0.013575,-0.068376,-0.176240,-0.154527,...,0.405810,0.424439,0.395601,-0.008603,-0.008169,-0.011917,-0.015779,-0.226950,0.051275,-0.172841


In [35]:
indexes.columns

Index(['label', 'sector', 'returns', 'distance_moving_avg', 'momentum_5d',
       'momentum_10d', 'momentum_20d', 'momentum_30d', 'momentum_120d',
       'momentum_250d', 'volatility_5d_ann', 'volatility_10d_ann',
       'volatility_15d_ann', 'volatility_20d_ann', 'volatility_25d_ann',
       'volatility_30d_ann', 'moving_avg_slope_3d', 'moving_avg_slope_5d',
       'moving_avg_slope_10d', 'moving_avg_slope_15d', 'distance_high_52w',
       'distance_low_52w', 'risk_adjusted_momentum'],
      dtype='object')

In [24]:
indexes["label_predict"] = final_model.predict(indexes[selected_features])
predict_proba = final_model.predict_proba(indexes[selected_features])


indexes["probability_lower"] = predict_proba[:, 0]
indexes["probability_neither"] = predict_proba[:, 1]
indexes["probability_upper"] = predict_proba[:, 2]

In [25]:
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

label_names = ["lower barrier", "neither barrier", "upper barrier"]
print("Balanced accuracy:", balanced_accuracy_score(indexes["label"], indexes["label_predict"]))
print("Confusion matrix:\n", confusion_matrix(indexes["label"], indexes["label_predict"], labels=[0, 1, 2]))
print(classification_report(
    indexes["label"],
    indexes["label_predict"],
    labels=[0, 1, 2],
    target_names=label_names,
    zero_division=0,
))

Balanced accuracy: 0.43861216412934795
Confusion matrix:
 [[3187  299  571]
 [ 914  881  457]
 [3469  301  609]]
                 precision    recall  f1-score   support

  lower barrier       0.42      0.79      0.55      4057
neither barrier       0.59      0.39      0.47      2252
  upper barrier       0.37      0.14      0.20      4379

       accuracy                           0.44     10688
      macro avg       0.46      0.44      0.41     10688
   weighted avg       0.44      0.44      0.39     10688



In [26]:
print("Accuracy :", accuracy_score(indexes["label"], indexes["label_predict"]))
print("Precision:", precision_score(indexes["label"], indexes["label_predict"], average='weighted'))
print("Recall   :", recall_score(indexes["label"], indexes["label_predict"], average='weighted'))
print("F1 Score :", f1_score(indexes["label"], indexes["label_predict"], average='weighted'))

Accuracy : 0.4375935628742515
Precision: 0.4375693172362802
Recall   : 0.4375935628742515
F1 Score : 0.3904945836567229


In [52]:
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
print(indexes["label"].value_counts(normalize=True))
print(classification_report(indexes["label"], indexes["label_predict"]))
print(confusion_matrix(indexes["label"], indexes["label_predict"]))
print(balanced_accuracy_score(indexes["label"], indexes["label_predict"]))

label
2    0.409712
0    0.379585
1    0.210704
Name: proportion, dtype: float64
              precision    recall  f1-score   support

           0       0.42      0.78      0.55      4057
           1       0.59      0.40      0.48      2252
           2       0.37      0.14      0.20      4379

    accuracy                           0.44     10688
   macro avg       0.46      0.44      0.41     10688
weighted avg       0.44      0.44      0.39     10688

[[3170  302  585]
 [ 900  899  453]
 [3455  310  614]]
0.4402603041337437
